# I. Overview

## Environment and install package

In [1]:
! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain

Defaulting to user installation because normal site-packages is not writeable
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp314-cp314-win_amd64.whl.metadata (21 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached watchfiles-1.2.0-cp314-cp314-win_amd64.whl.metadata (5.0 kB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ------------- -------------------------- 0.8/2.4 MB 7.7 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 8.0 MB/s  0:00:00
Using cached httpx_sse-0.4.3-py3-none-any.whl (9.0 kB)
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 26.8 MB/s  0:00:00
Using cached langchain_text_splitters-1.1.2-py3-none-any.whl (35 kB)
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-otlp-proto-http 1.42.1 requires opentelemetry-exporter-otlp-proto-common==1.42.1, but you have opentelemetry-exporter-otlp-proto-common 1.43.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.42.1 requires opentelemetry-proto==1.42.1, but you have opentelemetry-proto 1.43.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.42.1 requires opentelemetry-sdk~=1.42.1, but you have opentelemetry-sdk 1.43.0 which is incompatible.


In [18]:
!pip install -U langchain langchainhub langchain-classic

Defaulting to user installation because normal site-packages is not writeable
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.10
    Uninstalling langchain-1.3.10:
      Successfully uninstalled langchain-1.3.10


In [12]:
# Langsmith tracing and monitoring
import os
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
# os.environ['LANGCHAIN_API_KEY'] = <key>
# os.environ['OPENAI_API_KEY'] = <your-api-key>

In [82]:
from dotenv import load_dotenv
load_dotenv(".env")
# os.getenv("LANGCHAIN_API_KEY")
# os.getenv("OPENAI_API_KEY")
# os.getenv('GROQ_API_KEY')

True

## Library

In [88]:
# Import lib
import bs4
from langchain_classic import hub
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter # Dùng cho split
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate

from langchain_groq import ChatGroq

## Indexing 

### Load documents

In [ ]:
# Load documents: access URL -> parse HTML code by BeautifulSoup
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
# print(loader.load())
docs = loader.load()
print(len(docs))
docs[0]

1


Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='\n\n      LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview#\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes

In [43]:
print(len(docs[0].page_content))
print(docs[0].page_content)

43047


      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:

Planning

Subgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.
Reflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the quality of final results.


Memory

Short-term memo

### Split

In [34]:
# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
splits

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview#\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refi

In [56]:
print(len(splits))

63


### Embedding

In [ ]:
# Embedding (Using chromadb): OpenAIEmbeddings -> OpenAI Embeddings API -> Embedding Model (OpenAI)
embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vector_store = Chroma.from_documents(documents=splits, embedding=embedding)
vector_store

In [73]:
print(vector_store._collection.count())
print(vector_store._collection.get())

126
{'ids': ['02357835-9189-412e-8f7e-32697b71ff15', '9feceee6-7356-441d-8e70-a007c3e5938e', '4cc3ecf4-ff52-4ca7-8dfe-cab9b6e16143', 'c73f4a29-93e6-40b7-8e3f-10dda4ce48f5', '7d2bab6e-70ea-4dbf-b149-5c101a4fcceb', 'c05d8630-3625-45bb-ba65-0608793a8ee1', 'add1e4a5-7196-4ccd-82df-a2aa4c0bd853', 'cd624c86-8ef1-4b2a-a4e3-522fef9341e4', '6c77876b-0d6f-4217-a57a-d624eebfb4d9', '0c38b5ca-e30a-44d3-88cf-fa10b6358bdc', 'f41f5e0f-0fcc-46c6-9a82-b64bb6e9199b', '9f1fc218-57af-4084-8e79-a91454cf8dec', '516d9d11-e99e-483a-8d3f-d2f4b5c4fac0', 'eb49f886-0af4-4c3d-8fc0-f1a152b278fa', '4c08c48f-28a1-4c0e-9694-074a39aba149', '1111e3dc-bf3a-4f01-bc48-eda6a4e395c7', '1b7813b4-a568-4a1f-99fd-fd4a32e6984d', 'a3211d8c-e2d6-4a2d-94aa-de68ee232b62', '33190029-43e6-41fe-8071-ae32d760f40f', 'b1a076b5-bcb0-40c6-bbad-f93a760f5277', 'b95f394f-96cc-40bc-ab69-b8efb114502b', 'edeb2613-8f8e-4298-bf0a-8d10e75f3bef', 'f3919231-02e9-4cbd-bf59-4b32dfc88a08', '39024894-d200-442d-a5d6-c0689371e147', '7ded57ba-6f8d-4885-a65d-44

### Retrieval

In [78]:
print(os.getenv("GROQ_API_KEY"))

None


In [96]:
# retrieval
retriever = vector_store.as_retriever(
    search_kwargs={"k": 10}
)
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000002101B8F1BA0>, search_kwargs={'k': 10})

In [ ]:
# Prompt
# prompt = hub.pull(
#     "rlm/rag-prompt",
#     dangerously_pull_public_prompt=True
# )
prompt = ChatPromptTemplate.from_template("""
You are an assistant for question-answering tasks.
Use the following context to answer the question.
If you don't know the answer, say you don't know.
Keep the answer concise.

Question: {question}

Context: {context}

Based on context to answer
                                          
Answer:

""")

# LLM
llm = ChatGroq(
    model_name="openai/gpt-oss-20b", 
    temperature=0
)

# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()} # RunnablePassthrough is in langchain
    | prompt
    | llm
    | StrOutputParser() # lib of langchain
)

# Question
response = rag_chain.invoke("What is Task Decomposition?")

In [100]:
response

'Task decomposition is the process of breaking a complex task into a sequence of smaller, manageable sub‑tasks or steps. It can be done by prompting an LLM to list steps or subgoals, by using task‑specific instructions, or by incorporating human input. In some approaches, an LLM translates the problem into a formal representation (e.g., PDDL), an external planner generates a plan, and the plan is then converted back into natural language. The goal is to make difficult problems tractable by structuring them into clear, actionable parts.'